# 🚀 Huấn luyện Mô hình Gốc (Baseline RIFE)

In [ ]:
# 1. KẾT NỐI GOOGLE DRIVE & DI CHUYỂN VÀO DỰ ÁN
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM3_HK3_(2025-2026)/CT282_Deep Learning/PROJECT/RIFE-MinhTri/RIFE-Project"
os.chdir(PROJECT_DIR)
print(f"✅ Đang ở thư mục dự án: {os.getcwd()}")

In [ ]:
#@title 📥 2. NẠP DỮ LIỆU TỪ HUGGING FACE VÀO Ổ TẠM /content/ (KHÔNG CHIẾM DUNG LƯỢNG DRIVE)
!pip install -q datasets

from datasets import load_dataset
import os

TEMP_DATA_DIR = "/content/vimeo_triplet"
PROJECT_DIR = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM3_HK3_(2025-2026)/CT282_Deep Learning/PROJECT/RIFE-MinhTri/RIFE-Project"

if not os.path.exists(f"{TEMP_DATA_DIR}/sequences"):
    print("⏳ 1. Đang tải dataset bijinc/vimeo-90k-mini từ Hugging Face...")
    ds = load_dataset("bijinc/vimeo-90k-mini")

    print(f"⏳ 2. Đang giải nén dữ liệu vào ổ cứng tạm {TEMP_DATA_DIR}...")
    os.makedirs(f"{TEMP_DATA_DIR}/sequences", exist_ok=True)

    train_list = []
    test_list = []

    for split_name in ds.keys():
        print(f"   -> Đang xử lý tập {split_name} ({len(ds[split_name])} mẫu)...")
        for i, item in enumerate(ds[split_name]):
            folder = f"{split_name}/{i:05d}"
            os.makedirs(f"{TEMP_DATA_DIR}/sequences/{folder}", exist_ok=True)
            
            item['im1'].save(f"{TEMP_DATA_DIR}/sequences/{folder}/im1.png")
            item['im2'].save(f"{TEMP_DATA_DIR}/sequences/{folder}/im2.png")
            item['im3'].save(f"{TEMP_DATA_DIR}/sequences/{folder}/im3.png")
            
            if split_name == 'train':
                train_list.append(folder)
            else:
                test_list.append(folder)

    with open(f"{TEMP_DATA_DIR}/tri_trainlist.txt", "w") as f:
        f.write("\n".join(train_list))

    with open(f"{TEMP_DATA_DIR}/tri_testlist.txt", "w") as f:
        f.write("\n".join(test_list if test_list else train_list[:100]))
    print("✅ Đã tạo xong dữ liệu tại ổ tạm /content/!")

# 3. Tạo lối tắt từ Google Drive trỏ sang ổ tạm
!rm -rf "/content/drive/MyDrive/COMPUTER SCIENCE/NAM3_HK3_(2025-2026)/CT282_Deep Learning/PROJECT/RIFE-MinhTri/RIFE-Project/vimeo_triplet"
!ln -s /content/vimeo_triplet "/content/drive/MyDrive/COMPUTER SCIENCE/NAM3_HK3_(2025-2026)/CT282_Deep Learning/PROJECT/RIFE-MinhTri/RIFE-Project/vimeo_triplet"
print("🔗 Đã liên kết lối tắt thành công! Google Drive hoàn toàn nhẹ (0 MB data).")

In [ ]:
#@title 🏃 3. Bắt đầu Huấn luyện Baseline (PReLU gốc)
EPOCHS = 40 #@param {type:"integer"}
BATCH_SIZE = 16 #@param {type:"integer"}
SAVE_DIR = "trained_model/baseline_prelu"

!python train.py \
    --model_type original \
    --batch_size {BATCH_SIZE} \
    --epoch {EPOCHS} \
    --save_dir {SAVE_DIR}